# HW5: Bigfoot sighting reports

This notebook makes the two plots for my Jekyll page. The data is the BFRO bigfoot reports file from the class data repo.

In [1]:
import pandas as pd
import altair as alt

In [2]:
bf = pd.read_csv("https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/bfro_reports_fall2022.csv")
bf.head()

,observed,location_details,county,state,season,title,latitude,longitude,date,number,...,precip_intensity,precip_probability,precip_type,pressure,summary,uv_index,visibility,wind_bearing,wind_speed,location
0,Ed L. was salmon fishing with a companion in P...,East side of Prince William Sound,Valdez-Chitina-Whittier County,Alaska,Fall,NaN,NaN,NaN,NaN,1261.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,heh i kinda feel a little dumb that im reporti...,"the road is off us rt 80, i dont know the exit...",Warren County,New Jersey,Fall,NaN,NaN,NaN,NaN,438.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,I was on my way to Claremont from Lebanon on R...,Close to Claremont down 120 not far from Kings...,Sullivan County,New Hampshire,Summer,Report 55269: Dawn sighting at Stevens Brook o...,43.41549,-72.33093,2016-06-07,55269.0,...,0.001,0.7,rain,998.87,Mostly cloudy throughout the day.,6.0,9.70,262.0,0.49,POINT(-72.33093000000001 43.415490000000005)
3,I was northeast of Macy Nebraska along the Mis...,Latitude & Longitude : 42.158230 -96.344197,Thurston County,Nebraska,Spring,Report 59757: Possible daylight sighting of a ...,42.15685,-96.34203,2018-05-25,59757.0,...,0.000,0.0,NaN,1008.07,Partly cloudy in the morning.,10.0,8.25,193.0,3.33,POINT(-96.34203000000001 42.15685)
4,"While this incident occurred a long time ago, ...","Ward County, Just outside of a the Minuteman T...",Ward County,North Dakota,Spring,Report 751: Hunter describes described being s...,48.25422,-101.31660,2000-04-21,751.0,...,NaN,NaN,rain,1011.47,Partly cloudy until evening.,6.0,10.00,237.0,11.14,POINT(-101.3166 48.254220000000004)


In [3]:
bf.columns

Index(['observed', 'location_details', 'county', 'state', 'season', 'title',
       'latitude', 'longitude', 'date', 'number', 'classification', 'geohash',
       'temperature_high', 'temperature_mid', 'temperature_low', 'dew_point',
       'humidity', 'cloud_cover', 'moon_phase', 'precip_intensity',
       'precip_probability', 'precip_type', 'pressure', 'summary', 'uv_index',
       'visibility', 'wind_bearing', 'wind_speed', 'location'],
      dtype='str')

## Clean up the dates and seasons

In [4]:
# turn the date strings into real dates and get the year
bf['date'] = pd.to_datetime(bf['date'], errors='coerce')
bf['year'] = bf['date'].dt.year

# some reports have no date
bf['year'].isna().sum()

np.int64(950)

In [5]:
bf['season'].value_counts()

season
Summer     1765
Fall       1413
Spring      767
Winter      711
Unknown      91
Name: count, dtype: int64

In [6]:
bf['classification'].value_counts()

classification
Class B    2370
Class A    2347
Class C      30
Name: count, dtype: int64

In [7]:
# keep reports with a year and a known season, from 1950 on
bf_years = bf[bf['year'].notna() & (bf['season'] != 'Unknown')].copy()
bf_years['year'] = bf_years['year'].astype(int)
bf_years = bf_years[bf_years['year'] >= 1950]
len(bf_years)

3722

In [8]:
# count reports for each year, season and class
yearly = bf_years.groupby(['year', 'season', 'classification']).size().reset_index(name='reports')
yearly.head()

,year,season,classification,reports
0,1950,Spring,Class B,1
1,1950,Summer,Class A,2
2,1952,Summer,Class A,1
3,1953,Fall,Class A,1
4,1953,Summer,Class A,1


## Plot 1: reports per year with a brush that filters a season bar chart

In [9]:
brush = alt.selection_interval(encodings=['x'])

lines = alt.Chart(yearly).mark_line().encode(
    alt.X('year:Q', title='Year', axis=alt.Axis(format='d')),
    alt.Y('sum(reports):Q', title='Number of reports'),
    alt.Color('classification:N', title='Class')
).properties(
    width=500,
    height=200,
    title='Bigfoot reports per year'
).add_params(
    brush
)

bars = alt.Chart(yearly).mark_bar().encode(
    alt.X('season:N', sort=['Spring', 'Summer', 'Fall', 'Winter'], title='Season'),
    alt.Y('sum(reports):Q', title='Number of reports'),
    alt.Color('classification:N', title='Class')
).properties(
    width=500,
    height=200,
    title='Reports by season for the selected years'
).transform_filter(
    brush
)

chart1 = lines & bars
chart1

In [10]:
myJekyllDir = '../assets/json/'
chart1.save(myJekyllDir + 'bigfoot_timeline_seasons.json')

## Plot 2: heatmap of states and seasons with a dropdown

In [11]:
# top 20 states by number of reports
seasons = bf[bf['season'] != 'Unknown']
top_states = seasons['state'].value_counts().head(20).index

state_season = seasons[seasons['state'].isin(top_states)]
state_season = state_season.groupby(['state', 'season', 'classification']).size().reset_index(name='reports')
state_season.head()

,state,season,classification,reports
0,Alabama,Fall,Class A,13
1,Alabama,Fall,Class B,8
2,Alabama,Spring,Class A,7
3,Alabama,Spring,Class B,7
4,Alabama,Summer,Class A,22


In [12]:
dropdown = alt.binding_select(options=[None, 'Class A', 'Class B', 'Class C'],
                              labels=['All', 'Class A', 'Class B', 'Class C'],
                              name='Report class: ')
selection = alt.selection_point(fields=['classification'], bind=dropdown)

chart2 = alt.Chart(state_season).mark_rect().encode(
    alt.X('season:O', sort=['Spring', 'Summer', 'Fall', 'Winter'], title='Season'),
    alt.Y('state:N', sort='-color', title='State'),
    alt.Color('sum(reports):Q', title='Reports', scale=alt.Scale(scheme='orangered')),
    tooltip=['state', 'season', 'sum(reports)']
).properties(
    width=300,
    height=450,
    title='Bigfoot reports by state and season'
).add_params(
    selection
).transform_filter(
    selection
)

chart2

In [13]:
chart2.save(myJekyllDir + 'bigfoot_state_season_heatmap.json')